<a href="https://colab.research.google.com/github/tciodaro/GRLCDDR1C1-N2-L2-PB/blob/main/hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ajuste de Hiper-parâmetros

Durante o treinamento de modelos de ML é comum o teste sistemático de hiper-parâmetros, ou seja, os valores utilizados nos parâmetros do modelo. Esses ajustes têm como objetivo identificar os valores de parâmetros que conseguem atingir os melhores índices de generalização.

Nesse notebook, vamos testar 2 tipos de testes:
- Grid-search: método que testa a permutação de valores de pré-definidos de parâmetros.
- Random-search: método que modela os parâmetros de acordo com distribuições conhecidas de valores (Gaussianas, uniformes...). Diferentemente do grid-search, esse método sorteia diferentes valores para testar segundo as distribuições escolhidas para cada parâmetro.

## Overview

### Bases de dados

Vamos utilizar duas bases de dados:

- Qualidade de vinhos
- Categorias de textos

### Modelos

Vamos testar e ajustar os parâmetros de 2 diferentes modelos.

- Nearest Neighbors (parâmetro k)
- Decision Trees (parâmetro max_depth)


In [ ]:

import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

fname = 'drive/MyDrive/Curso Projeto de Bloco: Inteligência Artificial e Machine Learning (GRLCDDR1C1-N2-L2)/dataset_vinhos.csv'

target_col = 'target'
target_col_label='target_label'

cat_cols = ['type']


Mounted at /content/drive


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


wine = pd.read_csv(fname, sep=';')

wine['type'] = wine['type'].map({
    'white':0,
    'red': 1,
})


# Re-defining these variables as their original defining cell was not executed
target_col = 'target'
target_col_label='target_label'

# Separate features (X) and target (y)
y = wine[target_col]
X = wine.drop(columns=[target_col, target_col_label])

# Split data into development and validation sets (80/20 ratio)
X_train, X_tst, y_train, y_tst = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Data splitting complete.")
print(f"X_train shape: {X_train.shape}")
print(f"X_tst shape: {X_tst.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_tst shape: {y_tst.shape}")

Data splitting complete.
X_train shape: (4256, 12)
X_tst shape: (1064, 12)
y_train shape: (4256,)
y_tst shape: (1064,)


In [ ]:
KNeighborsClassifier?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

print("Preprocessing pipeline components (scaler, encoder, preprocessor) created.")
# Instantiate a KNeighborsClassifier with n_neighbors=10
knn_classifier = KNeighborsClassifier(n_neighbors=10)

# Create a Pipeline that combines the preprocessor and the kNN classifier
# This step needs to be re-executed after the preprocessor was updated.
model = Pipeline(steps=[('scaler', StandardScaler()),
                        ('pca', PCA(n_components=9)),
                        ('classifier', knn_classifier)])

param_grid = {
    'pca__n_components': [5, 10],
    'classifier__n_neighbors': [2,4,6,8,10],
}

classifier = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=['precision', 'recall', 'accuracy', 'roc_auc'],
    cv=10,
    refit = 'precision',
    return_train_score=True,
    n_jobs=-1
).fit(X_train, y_train)

print(classifier.best_params_)
print(classifier.best_score_)

Preprocessing pipeline components (scaler, encoder, preprocessor) created.
{'classifier__n_neighbors': 2, 'pca__n_components': 10}
0.8143464990168228


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

dtc_classifier = DecisionTreeClassifier()

# Create a Pipeline that combines the preprocessor and the kNN classifier
# This step needs to be re-executed after the preprocessor was updated.
model = Pipeline(steps=[('scaler', StandardScaler()),
                        ('pca', PCA(n_components=9)),
                        ('classifier', dtc_classifier)])

param_grid = {
    'pca__n_components': [5, 10],
    'classifier__max_depth': [2,4,6,8,10],
}

classifier = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=['precision', 'recall', 'accuracy', 'roc_auc'],
    cv=10,
    refit = 'precision',
    return_train_score=True,
    n_jobs=-1
).fit(X_train, y_train)

print(classifier.best_params_)
print(classifier.best_score_)

{'classifier__max_depth': 10, 'pca__n_components': 10}
0.75195451062718


In [ ]:
classifier.cv_results_.keys()

dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_classifier__n_neighbors', 'param_pca__n_components', 'params', 'split0_test_precision', 'split1_test_precision', 'split2_test_precision', 'split3_test_precision', 'split4_test_precision', 'split5_test_precision', 'split6_test_precision', 'split7_test_precision', 'split8_test_precision', 'split9_test_precision', 'mean_test_precision', 'std_test_precision', 'rank_test_precision', 'split0_train_precision', 'split1_train_precision', 'split2_train_precision', 'split3_train_precision', 'split4_train_precision', 'split5_train_precision', 'split6_train_precision', 'split7_train_precision', 'split8_train_precision', 'split9_train_precision', 'mean_train_precision', 'std_train_precision', 'split0_test_recall', 'split1_test_recall', 'split2_test_recall', 'split3_test_recall', 'split4_test_recall', 'split5_test_recall', 'split6_test_recall', 'split7_test_recall', 'split8_test_recall', 'split9_test_recall', 'mea

In [ ]:
classifier.best_params_

{'classifier__n_neighbors': 2, 'pca__n_components': 10}

In [ ]:
classifier.cv_results_['split4_test_roc_auc']

array([0.70576873, 0.75456387, 0.77879066, 0.79507926, 0.7946906 ,
       0.80822321, 0.79251172, 0.81577274, 0.80610322, 0.82154382])

In [ ]:
classifier.cv_results_['params']

[{'classifier__n_neighbors': 2, 'pca__n_components': 5},
 {'classifier__n_neighbors': 2, 'pca__n_components': 10},
 {'classifier__n_neighbors': 4, 'pca__n_components': 5},
 {'classifier__n_neighbors': 4, 'pca__n_components': 10},
 {'classifier__n_neighbors': 6, 'pca__n_components': 5},
 {'classifier__n_neighbors': 6, 'pca__n_components': 10},
 {'classifier__n_neighbors': 8, 'pca__n_components': 5},
 {'classifier__n_neighbors': 8, 'pca__n_components': 10},
 {'classifier__n_neighbors': 10, 'pca__n_components': 5},
 {'classifier__n_neighbors': 10, 'pca__n_components': 10}]

In [ ]:
RandomizedSearchCV?

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    'pca__n_components': [3, 5, 8, 10, 12],
    'classifier__n_neighbors': randint(1,20),
}

classifier = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    scoring='precision',
    cv=10,
    n_iter=10,
    refit = True,
    return_train_score=True,
    n_jobs=-1
).fit(X_train, y_train)

In [ ]:
classifier.cv_results_.keys()

dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_classifier__n_neighbors', 'param_pca__n_components', 'params', 'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score', 'split4_test_score', 'split5_test_score', 'split6_test_score', 'split7_test_score', 'split8_test_score', 'split9_test_score', 'mean_test_score', 'std_test_score', 'rank_test_score', 'split0_train_score', 'split1_train_score', 'split2_train_score', 'split3_train_score', 'split4_train_score', 'split5_train_score', 'split6_train_score', 'split7_train_score', 'split8_train_score', 'split9_train_score', 'mean_train_score', 'std_train_score'])

In [ ]:
classifier.cv_results_['params']

[{'classifier__n_neighbors': 19, 'pca__n_components': 8},
 {'classifier__n_neighbors': 5, 'pca__n_components': 5},
 {'classifier__n_neighbors': 11, 'pca__n_components': 3},
 {'classifier__n_neighbors': 11, 'pca__n_components': 3},
 {'classifier__n_neighbors': 19, 'pca__n_components': 12},
 {'classifier__n_neighbors': 12, 'pca__n_components': 8},
 {'classifier__n_neighbors': 15, 'pca__n_components': 3},
 {'classifier__n_neighbors': 16, 'pca__n_components': 3},
 {'classifier__n_neighbors': 1, 'pca__n_components': 10},
 {'classifier__n_neighbors': 12, 'pca__n_components': 5}]

In [ ]:
classifier.best_params_

{'classifier__n_neighbors': 12, 'pca__n_components': 8}